In [10]:
import re, sys, os
import logging
from pathlib import Path
from datetime import datetime
import torch
import torch_directml
from fastai.collab import *
from fastai.metrics import *
from fastai.data.external import *

torch._logging.set_logs(all=logging.WARNING)

In [11]:
print("="*80)
print("DIRECTML FULL DEBUG LOG STARTED")
print(f"torch version: {torch_directml.torch.__version__}")
print(f"Device: {torch_directml.device()}")
print(f"GPU: {torch_directml.device_name(0)}")
print("="*80)

DIRECTML FULL DEBUG LOG STARTED
torch version: 2.4.1+cpu
Device: privateuseone:0
GPU: Qualcomm(R) Adreno(TM) X1-85 GPU 


In [12]:
# 1. Detect DirectML device (no global default!)
dml = torch_directml.device()

## Local dataset cache

To avoid re-downloading the same data across runs, we save the dataset into a local `data/` folder inside the repo. If the data already exists, `untar_data` will reuse it rather than downloading again.

In [ ]:
local_data_path = Path('data')
local_data_path.mkdir(parents=True, exist_ok=True)
path = untar_data(URLs.ML_SAMPLE, dest=local_data_path)
print(f'FastAI dataset path: {path}')

In [14]:
dls = CollabDataLoaders.from_csv(
    path / 'ratings.csv', bs=64, device=dml,
    verbose=True, num_workers=0
)

Setting up after_item: Pipeline: 
Setting up before_batch: Pipeline: 
Setting up after_batch: Pipeline: ReadTabBatch


In [15]:
# For example, accuracy within a threshold
def rating_accuracy(inp, targ):
    """Percentage of predictions within 0.5 of actual rating"""
    return ((inp - targ).abs() < 0.5).float().mean()

In [16]:
learn = collab_learner(
    dls, y_range=(0.5,5.5), metrics=[rmse]
).to_fp16(enabled=False)
learn.to(dml)
""

''

In [17]:
# learn.fit_one_cycle instead of learn.fine_tune
# because we do not have a pretrained model for tabular data
learn.fit_one_cycle(20, lr_max=5e-3)

c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


epoch,train_loss,valid_loss,_rmse,time
0,1.570883,1.525260,1.235014,00:01
1,1.445874,1.280076,1.131404,00:01
2,0.960031,0.672664,0.820161,00:01
3,0.647702,0.647610,0.804742,00:01
4,0.541855,0.633529,0.795945,00:01
5,0.448270,0.654921,0.809272,00:01
6,0.354927,0.650218,0.806361,00:01
7,0.280351,0.658868,0.811707,00:01
8,0.233564,0.673741,0.820817,00:01
9,0.182844,0.687246,0.829003,00:01


In [18]:
learn.show_results()

c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


,userId,movieId,rating,rating_pred
0,25.0,44.0,2.0,2.270431
1,92.0,94.0,4.5,3.187332
2,56.0,7.0,4.0,4.370983
3,87.0,54.0,5.0,4.575851
4,1.0,99.0,2.0,2.554615
5,35.0,40.0,2.0,3.035737
6,85.0,96.0,5.0,4.783485
7,10.0,96.0,3.0,4.323616
8,67.0,96.0,3.5,3.403266


## Benchmark: CPU vs DirectML Adreno GPU

Now we run a small benchmark for the same collab model on both CPU-only and the DirectML Adreno GPU. This lets us compare training runtime and capture where work is executed.

In [19]:
import time
import pandas as pd

results = []

# GPU benchmark using DirectML
print('GPU device:', torch_directml.device_name(0))
print('DirectML available:', torch_directml.is_available())

gpu_device = torch_directml.device()
dls_gpu = CollabDataLoaders.from_csv(
    path / 'ratings.csv', bs=64, device=gpu_device,
    verbose=True, num_workers=0
)
learn_gpu = collab_learner(
    dls_gpu, y_range=(0.5,5.5), metrics=[rmse]
).to_fp16(enabled=False)
learn_gpu.to(gpu_device)

n_epochs = 5
start = time.perf_counter()
learn_gpu.fit_one_cycle(n_epochs, lr_max=5e-3)
gpu_time = time.perf_counter() - start
print(f'GPU DirectML run time: {gpu_time:.2f}s')
results.append({
    'setup': 'DirectML Adreno GPU',
    'epochs': n_epochs,
    'duration_s': gpu_time,
    'notes': 'DirectML device; some optimizer ops may fallback to CPU'
})

GPU device: Qualcomm(R) Adreno(TM) X1-85 GPU 
DirectML available: True
Setting up after_item: Pipeline: 
Setting up before_batch: Pipeline: 
Setting up after_batch: Pipeline: ReadTabBatch


c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


epoch,train_loss,valid_loss,_rmse,time
0,1.453052,1.193997,1.092702,00:01
1,0.814320,0.698996,0.836060,00:01
2,0.577578,0.663238,0.814394,00:01
3,0.483797,0.652532,0.807794,00:01
4,0.442836,0.650358,0.806448,00:01


GPU DirectML run time: 5.41s


In [20]:
# CPU-only benchmark
cpu_device = torch.device('cpu')
dls_cpu = CollabDataLoaders.from_csv(
    path / 'ratings.csv', bs=64, device=cpu_device,
    verbose=True, num_workers=0
)
learn_cpu = collab_learner(
    dls_cpu, y_range=(0.5,5.5), metrics=[rmse]
).to_fp16(enabled=False)
learn_cpu.to(cpu_device)

start = time.perf_counter()
learn_cpu.fit_one_cycle(n_epochs, lr_max=5e-3)
cpu_time = time.perf_counter() - start
print(f'CPU run time: {cpu_time:.2f}s')
results.append({
    'setup': 'CPU-only',
    'epochs': n_epochs,
    'duration_s': cpu_time,
    'notes': 'CPU-only baseline'
})

Setting up after_item: Pipeline: 
Setting up before_batch: Pipeline: 
Setting up after_batch: Pipeline: ReadTabBatch


c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
c:\Users\Said\k3sh4v_practicaldeeplearningconsumergpu\.venv\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


epoch,train_loss,valid_loss,_rmse,time
0,1.473359,1.154665,1.074553,00:00
1,0.823177,0.682922,0.826391,00:00
2,0.587035,0.660131,0.812485,00:00
3,0.497282,0.642440,0.801523,00:00
4,0.468803,0.640383,0.800239,00:00


CPU run time: 2.31s


In [21]:
df = pd.DataFrame(results)
df['duration_min'] = df['duration_s'] / 60

df

,setup,epochs,duration_s,notes,duration_min
0,DirectML Adreno GPU,5,5.408351,DirectML device; some optimizer ops may fallback to CPU,0.090139
1,CPU-only,5,2.306813,CPU-only baseline,0.038447


## Summary

This section compares the CPU-only baseline against the DirectML Adreno GPU setup. The left column is the CPU run and the right column is the GPU run. The metrics include total training time and any notes about fallback behavior.

- `DirectML Adreno GPU`: uses `torch-directml` and your Adreno device. Some unsupported backend operators may fall back to CPU.
- `CPU-only`: uses the same FastAI collab model and data entirely on CPU.

Use this comparison to judge whether the GPU path provides a meaningful speed advantage for your workflow.